### dependencies & globals


In [26]:
import requests
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import numpy as np

# county_selection = [
#     "Fulton",
#     "Cobb",
#     "DeKalb",
#     "Fayette",
#     "Gwinnett",
#     "Cherokee",
#     "Douglas",
#     "Forsyth",
#     "Clayton",
#     "Henry",
#     "Rockdale"
# ]

### API call to get GDOT projects


In [1]:
# depencencies
import requests
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import numpy as np

# Define the base URL of the MapServer layer
base_url = "https://rnhp.dot.ga.gov/hosting/rest/services/GDOT_Public_Outreach/Hub_Project_Search/MapServer/2/query"

# Define query parameters
params = {
    # filter by construction status to include both under construction and pre-construction
    "where": "CONSTRUTION_STATUS_DERIVED LIKE '%'",
    "outFields": "*",  # Get all fields
    "f": "geojson",  # Request data in GeoJSON format
    "returnGeometry": "true",  # Include geometry data
    "resultOffset": 0,  # Start from the first record
    "resultRecordCount": 1000  # Get 1000 records
}

# list to store each batch of GeoDataFrames
gdfs = []

while True:

    # Make the GET request
    response = requests.get(base_url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        batch_gdf = gpd.GeoDataFrame.from_features(data["features"])

        # if no more data is returned, break the loop
        if batch_gdf.empty:
            break

        gdfs.append(batch_gdf)

        print(
            f"Fetched {len(batch_gdf)} records (offset: {params['resultOffset']})")

        # increase offset by the batch size to fetch the next chunk
        params["resultOffset"] += params["resultRecordCount"]

    else:
        print(
            f"❌ Error fetching data: {response.status_code} - {response.text}")
        break

# Filter out empty GeoDataFrames
gdfs = [gdf.dropna(axis=1, how='all') for gdf in gdfs if not gdf.empty]

# Combine all batches into a single GeoDataFrame
if gdfs:
    gdot_projects = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

else:
    print("❌ No data retrieved.")

# drop any columns that ONLY contain NaN values
gdot_projects = gdot_projects.dropna(axis=1, how="all")

# # geographic filter: just get those in the metro
# gdot_projects = gdot_projects[gdot_projects['PROJECT_COUNTIES'].str.contains(
#     '|'.join(county_selection),
#     na=False)]

# replace spaces in the 'CONSTRUTION_STATUS_DERIVED' column with hyphens
gdot_projects['CONSTRUTION_STATUS_DERIVED'] = gdot_projects['CONSTRUTION_STATUS_DERIVED'].str.replace(
    " ", "-")
gdot_projects['CONSTRUCTION_STATUS_DERIVED'] = gdot_projects['CONSTRUTION_STATUS_DERIVED'].str.strip()

# if the 'CONTRACT_DESCRIPTION' column is empty, replace it with the 'SHORT_DESCR' column
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].fillna(
    gdot_projects['SHORT_DESCR'])

# define a list of sub-strings to keep in all caps; everything else in the 'SHORT_DESCR' column should be converted to title case
keep_all_caps = ["SR ", "CO ", "CR ", "US ", "RD", "SO ",
                 "CS ", "MI ", "SE ", "NE ", "SW ", "NW ",
                 "NS ", "CSX", "CCTV", "-TIA", " - TIA", "LCI", "GRTA"]

gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.title()
for sub_str in keep_all_caps:
    gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
        sub_str.title(), sub_str)

gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Brdg", "Bridge")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Mtn ", "Mountain ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Ii", "Phase II")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Ph Ii", "Phase II")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase IIi", "Phase III")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Ph IIi", "Phase III")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Iv", "Phase IV")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Vi", "Phase VI")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Resf ", "Resurface ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Rsrf ", "Resurface ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Shldr ", "Shoulder ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Recnst", "Reconstruction")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Cnst ", "Construction ")
gdot_projects['CONTRACT_DESCRIPTION'] = gdot_projects['CONTRACT_DESCRIPTION'].str.replace(
    "Fm ", "From ")


# filter out records that have either 'WPH' (completed) or 'REJ' (rejected) in the 'CONST_STAT_CD' column
gdot_projects = gdot_projects[~gdot_projects['CONST_STAT_CD'].isin([
                                                                   'WPH', 'REJ'])]

# rename columns
gdot_projects = gdot_projects.rename(columns={
    'LAST_REFRESH_DTTM': 'Last_refresh',
    'PROJECT_COUNTIES': 'Project_counties',
    'PROJ_ID': 'Project_ID',
    'CONTRACT_DESCRIPTION': 'Project_description',
    'CONTRACTOR_NAME': 'Contractor_name',
    'IS_TIA_PROJECT': 'Is_TIA_project',
    'CONSTRUTION_STATUS_DERIVED': 'CONSTRUCTION_STATUS_DERIVED'
})

# drop unneeded columns
gdot_projects = gdot_projects.drop(columns=[
    'OBJECTID',
    'PRIORITY_CD',
    'SOURCE_OF_CONSTRUCTION_DATES',
    'CONTRACT_ID',
    'CONSTRUTION_STATUS_DERIVED_RSN',
    'PAYMENT_PERCENT_COMPLETE',
    'ESRI_OID',
    'SHORT_DESCR',
    'REC_STATUS',
    'LET_RESP_CD',
    'PRIORITY_CD_DESCR',
    'CONST_STAT_CD',
    'CONSTRUCTION_PERCENT_COMPLETE',
    'CURR_COMPLETION_DATE',
    'AWARD_DATE'
])

# create URL column
gdot_projects['Project_URL'] = 'https://www.dot.ga.gov/applications/geopi/Pages/Dashboard.aspx?ProjectId=' + \
    gdot_projects['Project_ID'].astype(str)

# spatially join to congressional districts
gdot_projects = gdot_projects.set_crs(epsg=4326)

districts = gpd.read_file(
    'data/congressional_districts/Congress-2023 shape.shp')
districts = districts.to_crs(epsg=4326)
districts = districts[[
    'DISTRICT',
    'geometry'
]]

gdot_projects = gpd.sjoin(
    gdot_projects,
    districts,
    how='left',
    predicate='intersects'
).drop(columns=['index_right'])

# convert 'DISTRICT' column to integer
gdot_projects['DISTRICT'] = gdot_projects['DISTRICT'].astype(int)

# drop any duplicate columns
gdot_projects = gdot_projects.T.drop_duplicates().T

# Ensure all properties are JSON serializable
gdot_projects = gdot_projects.replace(
    {np.nan: None, np.inf: None, -np.inf: None})

# create a new GeoJSON feature collection with a unique integer ID for each feature
features = []
for i, row in gdot_projects.iterrows():

    feature = {
        'type': 'Feature',
        'id': i + 1,  # assign a unique integer ID
        'properties': row.drop('geometry').to_dict(),
        'geometry': row['geometry'].__geo_interface__
    }
    features.append(feature)

# create a new GeoJSON feature collection
collection = {
    'type': 'FeatureCollection',
    'features': features
}

# export the feature collection to a GeoJSON file
with open("data/GDOT_export.geojson", 'w') as f:
    json.dump(collection, f)

# export the GeoDataFrame to a CSV file
csv_export = gdot_projects.drop(columns=['geometry']).to_csv(
    'GDOT_export.csv', index=False)

# export timestamp file to be inserted in <div> on frontend
current_date = datetime.now().strftime("%B %d, %Y")
with open("data/current_date.txt", "w") as f:
    f.write(current_date)

# print status
print(f"✅ Successfully exported {len(gdot_projects):,} records!")

# Display final GeoDataFrame
gdot_projects

Fetched 1000 records (offset: 0)
Fetched 1000 records (offset: 1000)
Fetched 1000 records (offset: 2000)
Fetched 1000 records (offset: 3000)
Fetched 1000 records (offset: 4000)
Fetched 1000 records (offset: 5000)
Fetched 15 records (offset: 6000)
✅ Successfully exported 4,014 records!


,geometry,Project_ID,Project_counties,Is_TIA_project,Contractor_name,TPRO_PROJ_COMPLETE_DT,Project_description,TIME_STOPPED_DATE,SUBSTL_WORK_COMPL_DATE,CONSTRUCTION_STATUS_DERIVED,Last_refresh,Project_URL,DISTRICT
0,LINESTRING (-83.92952559898474 34.049612668536...,M006742,Gwinnett,No,None,None,SR 324 From SR 20 To SR 124,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,9
1,LINESTRING (-83.78039394215 34.591325121108206...,M006743,White,No,None,None,SR 115 From SR 11 To Habersham County Line,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,9
2,LINESTRING (-83.60692460212346 33.956312795935...,M006744,Barrow,No,None,None,SR 211 From SR 316 To SR 11,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,10
3,LINESTRING (-83.31563672100822 34.032002176249...,M006745,Madison,No,None,None,SR 8 From N Of SR 106 To SR 98,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,10
4,LINESTRING (-83.52634834511689 34.616635103747...,M006746,Habersham,No,None,None,SR 17 From SR 385 To White County Line,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6010,LINESTRING (-83.51259943206406 33.016939786186...,M006503,Jones,No,REEVES CONSTRUCTION CO,None,SR 22 - Pltmx Resurf,None,None,UNDER-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,8
6011,LINESTRING (-83.54736612451318 33.004271342533...,M006504,Jones,No,C W MATTHEWS CONTRACTING,None,"SR 18 - Mill, Plmx Rsrf",None,None,UNDER-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,8
6012,LINESTRING (-84.03149412326661 32.293498041361...,M006505,Macon,No,None,None,SR 49 From Randolph St To SR 90 & SR 90 From S...,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,2
6013,LINESTRING (-83.53060375265265 32.719203533969...,M006506,"Bibb , Houston , Monroe , Peach , Twiggs",No,None,None,I-16; I-75 & I-475 @ 8 Locs - Bridge Preservation,None,None,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,8


### playwright scraping


In [ ]:
INPUT_CSV = "GDOT_export.csv"

df = pd.read_csv(INPUT_CSV)
df[df['DISTRICT'] == 3]

,Project_ID,Project_counties,Is_TIA_project,Contractor_name,TPRO_PROJ_COMPLETE_DT,Project_description,TIME_STOPPED_DATE,SUBSTL_WORK_COMPL_DATE,CONSTRUCTION_STATUS_DERIVED,Last_refresh,Project_URL,DISTRICT
21,M006702,Troup,No,NaN,NaN,SR 1 & SR 14 From Ferrell Drive To SR 14 Spur,NaN,NaN,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
423,M006030,Muscogee,No,LOUIS-COMPANY LLC,NaN,Various Locations - Bridge Preservation,NaN,1.675210e+12,COMPLETED-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
433,M006351,"Harris , Muscogee",No,C W MATTHEWS CONTRACTING,NaN,"SR 85 - Mill, Inlay, Plmx Rsrf",1.725062e+12,1.724976e+12,COMPLETED-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
436,M006353,Troup,No,C W MATTHEWS CONTRACTING,NaN,"US 27/SR 1 - Milling, Inlay & Plmx Resf",1.711843e+12,1.709597e+12,COMPLETED-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
449,M006388,"Lamar , Spalding , Taylor , Upson",No,NaN,NaN,SR 3; SR 7 & SR 137 @ 7 Locs In Dist 3 - Bridg...,NaN,NaN,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
...,...,...,...,...,...,...,...,...,...,...,...,...
3888,M006466,Douglas,No,C W MATTHEWS CONTRACTING,NaN,"SR 5 - Mill, Plmx Rsrf",NaN,NaN,UNDER-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
3934,M006532,"Coweta , Fayette , Troup",No,NaN,NaN,Bridge Preservation @ 6 Locs In Coweta; Fayett...,NaN,NaN,PRE-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
3958,M006488,Lamar,No,WILLIAMS CONTRACTING CO,NaN,SR 7 Sb & Nb - Bridge Rehab,NaN,NaN,UNDER-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3
3977,M006493,Troup,No,C W MATTHEWS CONTRACTING,NaN,Kia Blvd & Kia Parkway - Pltmx Resurf,NaN,NaN,UNDER-CONSTRUCTION,1742374525000,https://www.dot.ga.gov/applications/geopi/Page...,3


In [ ]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

# Input and output file paths
INPUT_CSV = "GDOT_export.csv"
OUTPUT_CSV = "gdot_scraped_data.csv"

df = pd.read_csv(INPUT_CSV)
df = df[df['DISTRICT'] == 3]

# Get the total number of projects
total_projects = len(df)

# Initialize the project index
project_index = 0

# Create an empty list to store scraped data
all_data = []


async def scrape_project_data():
    global project_index

    # Start async Playwright
    async with async_playwright() as p:
        # Launch a browser (headless mode for speed)
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Loop through each project URL
        for index, row in df.iterrows():
            project_id = row["Project_ID"]
            project_url = row["Project_URL"]
            project_index += 1

            # Calculate the percentage
            percentage = (project_index / total_projects) * 100
            print(
                f"Scraping Project ID: {project_id} ({project_index} of {total_projects}, or {percentage:.1f}% complete)")

            # Open the project page
            await page.goto(project_url)

            # Select all rows inside the tbody
            rows = await page.query_selector_all("table.rgMasterTable tbody tr")

            # Wait for the table to load before scraping
            try:
                await page.wait_for_selector("table.rgMasterTable", timeout=15000)
            except:
                print(
                    f"Table not found for Project ID {project_id}. Skipping...")
                continue

            # ================================
            # ✅ Scrape Project Description
            # ================================
            try:
                # Get all rows in the ProjectDescriptionTable
                description_rows = await page.query_selector_all("table.ProjectDescriptionTable tbody tr")

                # Check if there is a second <tr> (index 1), then get its <td>
                if len(description_rows) >= 2:
                    second_row_cells = await description_rows[1].query_selector_all("td")
                    if second_row_cells:
                        project_description = (await second_row_cells[0].inner_text()).strip()
                    else:
                        project_description = "N/A"
                else:
                    project_description = "N/A"

            except Exception:
                project_description = "N/A"

            # ==========================================
            # ✅ Scrape Project Manager & Completion Date
            # ==========================================
            try:
                project_info_rows = await page.query_selector_all("table.ProjectInformationTable tbody tr")

                # Check if the 3rd row (index 2) exists and has 4 <td> elements
                if len(project_info_rows) >= 3:
                    third_row_cells = await project_info_rows[2].query_selector_all("td")
                    if len(third_row_cells) >= 4:
                        project_manager = (await third_row_cells[1].inner_text()).strip()
                        completion_date = (await third_row_cells[3].inner_text()).strip()
                    else:
                        project_manager, completion_date = "N/A", "N/A"
                else:
                    project_manager, completion_date = "N/A", "N/A"
            except Exception:
                project_manager, completion_date = "N/A", "N/A"

            # ================================
            # ✅ Scrape Lower Table
            # ================================

            # Get all rows from the table's <tbody> section
            rows = await page.query_selector_all("table.rgMasterTable tbody tr")

            for row in rows:
                cells = await row.query_selector_all("td")
                cell_data = [await cell.inner_text() for cell in cells]

                # Clean non-breaking spaces and weird characters
                clean_data = [cell.replace("\xa0", " ").replace(
                    "¬†", "").strip() for cell in cell_data]

                # Add Project_ID and Project_URL to the start of the row
                if len(clean_data) == 4:
                    all_data.append([
                        project_id, project_url, *clean_data,
                        project_description, project_manager, completion_date
                    ])

        # Close the browser after scraping
        await browser.close()


# Run the scraper
asyncio.run(scrape_project_data())

# Define the column headers
columns = [
    "Project_ID", "Project_URL", "Activity",
    "Program Year", "Cost Estimate", "Date of Last Estimate",
    "Project_description", "Project_manager", "Completion_date"
]

# Create DataFrame and drop duplicate/empty rows
clean_df = pd.DataFrame(all_data, columns=columns)

# Drop empty rows or unnecessary ones
clean_df = clean_df.replace("", pd.NA).dropna(how="all", subset=columns[2:])

# reformat the 'Cost Estimate' column
clean_df['Cost Estimate'] = clean_df['Cost Estimate'].fillna(0)
clean_df['Cost Estimate'] = clean_df['Cost Estimate'].str.replace(
    '$', '').str.replace(',', '').astype(float)

# Group by Project_ID, summing "Cost Estimate" while keeping the first occurrence of other columns
final_df = clean_df.groupby("Project_ID", as_index=False).agg(
    Project_URL=("Project_URL", "first"),
    Total_Cost=("Cost Estimate", "sum"),
    Project_manager=("Project_manager", "first"),
    Description=("Project_description", "first"),
)

# Save cleaned data to a new CSV
final_df.to_csv("GDOT_scraped_data_cleaned.csv", index=False)

print("✅ Scraping complete!")

Scraping Project ID: M006702 (1 of 295, or 0.3% complete)
Scraping Project ID: M006030 (2 of 295, or 0.7% complete)
Scraping Project ID: M006351 (3 of 295, or 1.0% complete)
Scraping Project ID: M006353 (4 of 295, or 1.4% complete)
Scraping Project ID: M006388 (5 of 295, or 1.7% complete)
Scraping Project ID: M006357 (6 of 295, or 2.0% complete)
Scraping Project ID: M006332 (7 of 295, or 2.4% complete)
Scraping Project ID: M006341 (8 of 295, or 2.7% complete)
Scraping Project ID: M006343 (9 of 295, or 3.1% complete)
Scraping Project ID: M006344 (10 of 295, or 3.4% complete)
Scraping Project ID: 0019227 (11 of 295, or 3.7% complete)
Scraping Project ID: 0019066 (12 of 295, or 4.1% complete)
Scraping Project ID: 0019524 (13 of 295, or 4.4% complete)
Scraping Project ID: 0019710 (14 of 295, or 4.7% complete)
Scraping Project ID: 0019531 (15 of 295, or 5.1% complete)
Scraping Project ID: 0019777 (16 of 295, or 5.4% complete)
Scraping Project ID: 0019783 (17 of 295, or 5.8% complete)
Scrapi

In [62]:
scraped_df = pd.read_csv("GDOT_scraped_data_cleaned.csv")

alt_data = pd.read_csv("GDOT_export.csv")

d3_projects = pd.merge(
    scraped_df,
    alt_data,
    how="left",
    on="Project_ID"
).drop(columns=["Project_URL_y"]).rename(columns={
    "Project_URL_x": "Project_URL",
    "Total_Cost": "Project_cost",
    "Project_description": "Description_short",
    "CONSTRUCTION_STATUS_DERIVED": "Construction_status",
})

d3_projects = d3_projects[d3_projects['DISTRICT'] == 3]

d3_projects = d3_projects[[
    'Project_ID',
    'Construction_status',
    'Project_manager',
    'Project_cost',
    'Project_counties',
    'Description',
    'Description_short',
    'Is_TIA_project',
    'Project_URL'
]]

d3_projects.to_csv('GDOT_Distrct3_projects.csv', index=False)